# 1. Verimlilik ölçümü

Her (model, seviye) çifti için: yükleme süresi, model ayak izi, tepe GPU belleği,
üretim hızı (token/sn), Türkçe metinlerde perplexity ve üç örnek soruya verilen cevap.

Bütün ayarlar `kod/ortak.py`'den gelir; bu notebook hiçbir üretim ayarı tanımlamaz
(vize ölçütü 4: tüm koşullarda aynı prompt ve decoding).

**Donanım notu:** FP16 ayak izi yaklaşık 15 GB. Colab'ın ücretsiz tek T4'ü (16 GB)
KV önbellekle birlikte buna yetmiyor. FP16 koşusunu Kaggle'da **GPU T4 x2** ile
yapın, kuantize seviyeleri Colab'da çalışır. Her koşu kendi CSV'sini yazdığı için
sonuçlar sonradan birleşir (`05_analiz.ipynb`).

Başlamadan önce ortamı kontrol edin: `!python kod/kontrol.py`

In [ ]:
!pip install -q -U "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "datasets>=2.20"

In [ ]:
import os
import sys
import time

import pandas as pd

# Depo kökünü bul: Colab'da /content/proje ya da /content, Kaggle'da /kaggle/...
for _kok in ["/content/proje", "/content", "/kaggle/working/proje", "/kaggle/working", "."]:
    if os.path.isdir(os.path.join(_kok, "kod")):
        sys.path.insert(0, os.path.join(_kok, "kod"))
        break
else:
    raise RuntimeError(
        "kod/ klasörü bulunamadı. Depoyu klonlayın:\n"
        "  !git clone https://github.com/<kullanici>/<depo>.git /content/proje"
    )

import ortak as O

O.set_seed()
O.hf_login()
print(O.env_report())
print("Çıktı klasörü:", O.OUT_DIR)

## Koşu listesi

`MODEL_KEY` ve `RUN_VARIANTS` bu notebook'un tek parametreleri. Ortamın belleğine
sığmayan seviyeyi listeden çıkarın; tamamlanmış koşular otomatik atlanır.

`gptq4` ve `awq4` için ağırlıkların önceden üretilmiş olması gerekir:
`!python kod/kuantize_gptq_awq.py --model trendyol --yontem gptq4`

In [ ]:
MODEL_KEY = "trendyol"              # O.MODELS: trendyol, cosmos
RUN_VARIANTS = ["int8", "nf4"]      # Kaggle 2xT4'te: ["fp16", "int8", "nf4"]

print("Model:", O.MODELS[MODEL_KEY])
print("Seviyeler:", [O.VARIANT_LABELS[v] for v in RUN_VARIANTS])

In [ ]:
# Model dosyalarını önce indir, böylece ölçülen yükleme süresi indirmeyi içermez.
from huggingface_hub import snapshot_download

t0 = time.perf_counter()
snapshot_download(O.MODELS[MODEL_KEY], allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
print(f"İndirme/önbellek kontrolü: {time.perf_counter() - t0:.0f} sn")

tok = O.load_tokenizer(MODEL_KEY)
print("Sohbet şablonu var mı:", tok.chat_template is not None)

In [ ]:
for variant in RUN_VARIANTS:
    run = O.Run(MODEL_KEY, variant, "verimlilik")
    if run.done():
        print(f"{run.name}: zaten var, atlanıyor")
        continue

    print(f"\n===== {MODEL_KEY} / {O.VARIANT_LABELS[variant]} =====")
    O.free_gpu()
    O.reset_peak_memory()

    t0 = time.perf_counter()
    model = O.load_model(MODEL_KEY, variant)
    load_s = time.perf_counter() - t0
    footprint = model.get_memory_footprint() / 1024**3
    print(f"Yükleme {load_s:.0f} sn | ayak izi {footprint:.2f} GB")

    # Isınma: ilk çağrıdaki CUDA çekirdek derlemesi ölçüme karışmasın.
    O.generate(tok, model, "Merhaba!", max_new_tokens=8)

    speeds = []
    for _ in range(O.SPEED_REPEATS):
        _, n_tok, sec = O.generate(tok, model, O.SPEED_PROMPT,
                                   max_new_tokens=O.SPEED_TOKENS,
                                   min_new_tokens=O.SPEED_TOKENS)
        speeds.append(n_tok / sec)
    tok_per_s = sum(speeds) / len(speeds)
    print(f"Hız {tok_per_s:.1f} token/sn")

    ppl = O.perplexity(tok, model, O.PPL_TEXTS)
    print(f"Perplexity {ppl:.3f}")

    samples = []
    for prompt in O.SAMPLE_PROMPTS:
        answer, _, _ = O.generate(tok, model, prompt)
        samples.append({"model": MODEL_KEY, "variant": variant,
                        "prompt": prompt, "answer": answer})
        print(f"\nS: {prompt}\nC: {answer[:300]}")

    pd.DataFrame([{
        "model": MODEL_KEY, "variant": variant,
        "load_s": round(load_s, 1), "footprint_gb": round(footprint, 2),
        "peak_gpu_gb": round(O.peak_memory_gb(), 2),
        "tok_per_s": round(tok_per_s, 2), "perplexity": round(ppl, 3),
        "gpus": "|".join(O.gpu_names()),
    }]).to_csv(run.path(), index=False)
    pd.DataFrame(samples).to_csv(run.path("_ornekler.csv"), index=False,
                                 encoding="utf-8-sig")
    print(f"\nKaydedildi: {run.path()}")

    del model
    O.free_gpu()

In [ ]:
# Bu ortamda tamamlanan koşular
import glob

rows = [pd.read_csv(f) for f in sorted(glob.glob(f"{O.OUT_DIR}/*_verimlilik.csv"))]
if rows:
    display(pd.concat(rows, ignore_index=True))
else:
    print("Henüz sonuç yok.")